# Ultron — OCR experiment tracking and model selection (MLflow)

Every model we tested is logged as an **MLflow run**: which model, which settings, which test lines, which scores.
MLflow then gives a leaderboard, charts of how results evolved, and a **model registry** that marks which model is in production and why.

```
fixed test lines ──▶ model A ──┐
(CATMuS test)    ──▶ model B ──┼──▶ MLflow runs ──▶ leaderboard + charts ──▶ registry: "production" = TrOCR base
                 ──▶ model C ──┘   (params, metrics,
                                     predictions)
```

**Result of the 11 November contest (5 models, same 100 lines):** MEDUSA wins on accuracy (8.9 % CER vs 19.4 % for TrOCR) but needs 14 GB of GPU and 6x more time per line, and it was trained on CATMuS. TrOCR therefore stays in production and MEDUSA becomes the main candidate (see section 6).

**How to use it**
- Sections 1–6 need **no GPU** and run in about 2 minutes: they log all our past experiments, compare them and fill the registry.
- Section 7 (optional, GPU) evaluates a model again and adds it as a new run.
- Section 8 opens the MLflow web interface inside Colab.

## 1. Install

In [ ]:
%pip install -q mlflow matplotlib
import os
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
import mlflow
print("MLflow", mlflow.__version__)

## 2. Where the experiments are stored

Set `USE_DRIVE = True` to keep the MLflow database in your Google Drive, so it survives a Colab disconnection and the whole team can reuse it.

In [ ]:
import os
import mlflow
from mlflow import MlflowClient

USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/ultron-mlflow"
else:
    BASE_DIR = os.path.abspath("ultron-mlflow")
os.makedirs(BASE_DIR, exist_ok=True)

TRACKING_URI = f"sqlite:///{BASE_DIR}/mlflow.db"
mlflow.set_tracking_uri(TRACKING_URI)
EXPERIMENT = "ultron-ocr"
if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    mlflow.create_experiment(EXPERIMENT, artifact_location=f"{BASE_DIR}/artifacts")
experiment = mlflow.set_experiment(EXPERIMENT)
client = MlflowClient()
print("Experiments stored in", BASE_DIR)

## 3. Our past experiments

These are the results the team measured between 4 and 31 October 2026, written down once here.
The benchmark is always built from the **CATMuS Medieval test split**:

| Benchmark name | Lines |
| --- | --- |
| `catmus-test-buffer500` | first test rows, shuffled with a small buffer (only a few manuscripts) |
| `catmus-test-every50` | 1 line out of every 50 of the test split (300 lines, 5 languages) — our reference |
| `catmus-test-every50-third` | 1 line out of 3 of the reference set (100 lines, same 5 languages) — used for the contest |
| `catmus-test-first5` | the first 5 lines of the reference set (quick check only) |
| `catmus-page-bnf-esp36` | 12 consecutive lines of one Castilian manuscript stacked into a page |

In [ ]:
TROCR_BASE = "medieval-data/trocr-medieval-base"
TROCR_CURSIVA = "medieval-data/trocr-medieval-cursiva"
QWEN2_CATMUS = "medieval-data/qwen2-vl-2b-catmus"
QWEN3_CATMUS = "Qwen/Qwen3-VL-4B-Instruct + small-models-for-glam/Qwen3-VL-4B-catmus"
MEDUSA = "ENC-PSL/Medusa0.2Line-4B"
GREEDY = "greedy"
BEAMS = "beam search (num_beams=4, no_repeat_ngram_size=4)"

HISTORY = [
    # id, date, model, benchmark, lines, decoding, metrics, notes
    dict(hid="H01", date="2026-10-04", model=TROCR_BASE, benchmark="catmus-test-buffer500", lines=50,
         decoding=GREEDY, cer=0.109, notes="First test. Only 3 languages, all 14th century: not representative."),
    dict(hid="H02", date="2026-10-04", model=TROCR_BASE, benchmark="catmus-test-buffer500", lines=300,
         decoding=GREEDY, cer=0.141, notes="More lines but still the same few manuscripts."),
    dict(hid="H03", date="2026-10-04", model=TROCR_BASE, benchmark="catmus-test-every50", lines=300,
         decoding=GREEDY, cer=0.257, notes="Representative sample (5 languages, 10th-15th c.). 3 lines loop ('a a a ...')."),
    dict(hid="H04", date="2026-10-04", model=TROCR_BASE, benchmark="catmus-test-every50", lines=300,
         decoding=BEAMS, cer=0.191, notes="Beam search removes the repetition loops. Becomes the baseline."),
    dict(hid="H05", date="2026-10-14", model=TROCR_BASE, benchmark="catmus-test-every50", lines=300,
         decoding=BEAMS, cer=0.1883, wer=0.5442, notes="Re-run by a teammate with jiwer: confirms H04."),
    dict(hid="H06", date="2026-10-14", model=QWEN2_CATMUS, benchmark="catmus-test-every50", lines=300,
         decoding=GREEDY, cer=0.2674, wer=0.6980, prompt="Transcribe the text in this image exactly.",
         notes="Worse than TrOCR. Prompt differs from the training prompt ('Convert this image to text.')."),
    dict(hid="H07", date="2026-10-17", model=TROCR_CURSIVA, benchmark="catmus-test-first1", lines=1,
         decoding=BEAMS, cer=0.105, notes="Model first used in the app. One control line only."),
    dict(hid="H08", date="2026-10-17", model=TROCR_BASE, benchmark="catmus-test-first1", lines=1,
         decoding=BEAMS, cer=0.053, notes="Same control line: base is better, the app switches to base."),
    dict(hid="H09", date="2026-10-19", model=TROCR_BASE, benchmark="catmus-page-bnf-esp36", lines=12,
         decoding=BEAMS, cer=0.059, component="recognizer", notes="Each line given separately (perfect crops)."),
    dict(hid="H10", date="2026-10-19", model=TROCR_BASE, benchmark="catmus-page-bnf-esp36", lines=12,
         decoding=BEAMS, cer=0.101, component="pipeline", segmenter="profile",
         notes="Whole page through ocr.transcribe_image: 12/12 lines found, splitting costs about 4 points."),
    dict(hid="H11", date="2026-10-30", model=TROCR_BASE, benchmark="catmus-test-first5", lines=5,
         decoding=BEAMS, cer=0.063, wer=0.324, s_per_line=1.34, gpu_gb=1.4, notes="Quick check."),
    dict(hid="H12", date="2026-10-30", model=QWEN3_CATMUS, benchmark="catmus-test-first5", lines=5,
         decoding=GREEDY, cer=0.091, wer=0.382, s_per_line=11.14, gpu_gb=9.3,
         prompt="Transcribe the text shown in this image.", notes="Quick check."),
    dict(hid="H13", date="2026-10-30", model=MEDUSA, benchmark="catmus-test-first5", lines=5,
         decoding=GREEDY, cer=0.051, wer=0.294, s_per_line=10.56, gpu_gb=9.3,
         prompt="Transcribe the handwritten text in this line image. Output ONLY the transcription.",
         notes="Quick check: best of the three on 5 lines."),
    dict(hid="H14", date="2026-10-30", model=TROCR_BASE, benchmark="catmus-test-every50", lines=300,
         decoding=BEAMS, cer=0.191, wer=0.544, s_per_line=1.07, gpu_gb=1.4, notes="Same score as H04: reproducible."),
    dict(hid="H15", date="2026-10-31", model=TROCR_BASE, benchmark="catmus-test-every50-third", lines=100,
         decoding=BEAMS, cer=0.194, wer=0.589, s_per_line=1.05, gpu_gb=1.4, notes="Head-to-head with Qwen3-VL."),
    dict(hid="H16", date="2026-10-31", model=QWEN3_CATMUS, benchmark="catmus-test-every50-third", lines=100,
         decoding=GREEDY, cer=0.195, wer=0.558, s_per_line=7.9, gpu_gb=11.4,
         prompt="Transcribe the text shown in this image.", notes="Ties with TrOCR, 7x slower, 8x more GPU memory."),
    dict(hid="H17", date="2026-10-31", model=MEDUSA, benchmark="catmus-test-every50-third", lines=100,
         decoding=GREEDY, status="FAILED", s_per_line=10.6, gpu_gb=9.3,
         prompt="Transcribe the handwritten text in this line image. Output ONLY the transcription.",
         notes="Did not finish on free Colab (session lost while running)."),
]

# Contest of 11 November 2026: the 5 models on the same 100 lines, same session, same T4 GPU.
# For vision-language models, the one line above 300,000 pixels (5152x208) was shrunk to fit in GPU memory.
CONTEST_ID = "contest-100-2026-11-11"
CONTEST = [
    (TROCR_BASE, BEAMS, None, 0.194, 0.589, 1.07, 1.4, "Same score as H15: the benchmark is reproducible."),
    (TROCR_CURSIVA, BEAMS, None, 0.227, 0.661, 1.02, 1.4, "Worse than trocr-base, as on the control line."),
    (QWEN2_CATMUS, GREEDY, "Convert this image to text.", 0.222, 0.646, 2.00, 9.4,
     "With its training prompt: 22.2 % instead of 26.7 % (H06). The prompt matters."),
    (QWEN3_CATMUS, GREEDY, "Transcribe the text shown in this image.", 0.196, 0.558, 7.40, 13.6,
     "Same as H16: ties with TrOCR, 7x slower."),
    (MEDUSA, GREEDY, "Transcribe the handwritten text in this line image. Output ONLY the transcription.",
     0.089, 0.348, 6.71, 13.7,
     "Winner, half the errors of the others. Trained on CATMuS: test-split exclusion not yet confirmed."),
]
HISTORY += [dict(hid=None, date="2026-11-11", model=m,
                 benchmark="catmus-test-every50-third", lines=100, decoding=d, cer=cer, wer=wer,
                 s_per_line=s, gpu_gb=g, notes=n, contest=True, **({"prompt": p} if p else {}))
            for m, d, p, cer, wer, s, g, n in CONTEST]
print(len(HISTORY), "experiments, including the 5 runs of the contest")

In [ ]:
from datetime import datetime, timezone
from mlflow.entities import Metric, Param, RunTag

def short_name(model):
    return {TROCR_BASE: "trocr-base", TROCR_CURSIVA: "trocr-cursiva", QWEN2_CATMUS: "qwen2-vl-2b-catmus",
            QWEN3_CATMUS: "qwen3-vl-4b-catmus", MEDUSA: "medusa-0.2-4b"}.get(model, model)

for h in HISTORY:
    if h.get("contest"):
        h["hid"] = f"C-{short_name(h['model'])}"

previous = client.search_runs([experiment.experiment_id], max_results=1000)
already = {r.data.tags.get("history_id") for r in previous}
# runs already logged by the contest notebook (same Drive database) are not logged twice either
already |= {f"C-{r.data.tags.get('contest_model')}" for r in previous if r.data.tags.get("contest_id") == CONTEST_ID}
added = 0
for h in HISTORY:
    if h["hid"] in already:
        continue                                    # never log the same past experiment twice
    ts = int(datetime.fromisoformat(h["date"]).replace(hour=12, tzinfo=timezone.utc).timestamp() * 1000)
    run = client.create_run(experiment.experiment_id, start_time=ts, run_name=f'{h["hid"]} {short_name(h["model"])} {h["benchmark"]}')
    params = {"model_id": h["model"], "model": short_name(h["model"]), "benchmark": h["benchmark"],
              "lines": h["lines"], "decoding": h["decoding"], "component": h.get("component", "recognizer"),
              "segmenter": h.get("segmenter", "none (line images)"), "prompt": h.get("prompt", "none (TrOCR)")}
    metrics = {k: h[k] for k in ("cer", "wer", "s_per_line", "gpu_gb") if k in h}
    tags = {"history_id": h["hid"], "source": "historical", "date": h["date"], "notes": h["notes"],
            "mlflow.note.content": h["notes"]}
    if h.get("contest"):
        tags.update({"source": "contest", "contest_id": CONTEST_ID, "contest_model": short_name(h["model"])})
    client.log_batch(run.info.run_id,
                     metrics=[Metric(k, float(v), ts, 0) for k, v in metrics.items()],
                     params=[Param(k, str(v)) for k, v in params.items()],
                     tags=[RunTag(k, str(v)) for k, v in tags.items()])
    client.set_terminated(run.info.run_id, status=h.get("status", "FINISHED"), end_time=ts + 60_000)
    added += 1
print(f"{added} past experiments added ({len(HISTORY) - added} were already there)")

## 4. Leaderboard

All runs, then the two fair comparisons: same lines, same metric.

In [ ]:
import pandas as pd

def load_runs():
    df = mlflow.search_runs([experiment.experiment_id], max_results=1000)
    df = df[df["tags.source"] != "registry"]          # registry entries are not experiments
    keep = {"tags.history_id": "id", "tags.date": "date", "params.model": "model", "params.benchmark": "benchmark",
            "params.lines": "lines", "params.decoding": "decoding", "params.component": "component",
            "metrics.cer": "CER", "metrics.wer": "WER", "metrics.s_per_line": "s_per_line",
            "metrics.gpu_gb": "GPU_GB", "status": "status", "tags.notes": "notes"}
    df = df[[c for c in keep if c in df.columns]].rename(columns=keep)
    df["lines"] = pd.to_numeric(df["lines"])
    df["id"] = df["id"].fillna("C-" + df["model"])     # runs logged by the contest notebook
    return df.sort_values(["date", "id"]).reset_index(drop=True)

runs = load_runs()
fmt = {"CER": "{:.1%}", "WER": "{:.1%}", "s_per_line": "{:.2f}", "GPU_GB": "{:.1f}"}
display(runs.style.format(fmt, na_rep="–").hide(axis="index"))

contest = runs[runs.id.str.startswith("C-")].sort_values("CER").reset_index(drop=True)
contest.index += 1
print("\n=== Contest of 11 November 2026: 5 models, same 100 lines, same GPU ===")
display(contest[["model", "CER", "WER", "s_per_line", "GPU_GB", "notes"]].style.format(fmt, na_rep="–"))

board = runs[(runs.benchmark == "catmus-test-every50") & (runs.status == "FINISHED")].sort_values("CER")
print("\n=== Earlier runs on the 300-line set ===")
display(board[["id", "date", "model", "decoding", "CER", "WER"]].style.format(fmt, na_rep="–").hide(axis="index"))

## 5. How the results evolved

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(17, 6.5), gridspec_kw={"width_ratios": [3, 2]})
colors = {"trocr-base": "#2f6db5", "trocr-cursiva": "#8fa9c9", "qwen2-vl-2b-catmus": "#c2672b",
          "qwen3-vl-4b-catmus": "#7a4fa8", "medusa-0.2-4b": "#2a8a5a"}
label = {"trocr-base": "TrOCR", "trocr-cursiva": "TrOCR cursiva", "qwen2-vl-2b-catmus": "Qwen2-VL",
         "qwen3-vl-4b-catmus": "Qwen3-VL", "medusa-0.2-4b": "MEDUSA"}

# Left: every finished run on the reference benchmarks, in date order
ref = runs[runs.benchmark.isin(["catmus-test-every50", "catmus-test-every50-third"]) & (runs.status == "FINISHED")]
ref = pd.concat([ref[ref.id.str.startswith("H")], ref[~ref.id.str.startswith("H")].sort_values("CER")])
ax = axes[0]
for i, row in enumerate(ref.itertuples()):
    ax.bar(i, row.CER * 100, color=colors.get(row.model, "#888888"), width=0.7)
    ax.text(i, row.CER * 100 + 0.4, f"{row.CER * 100:.1f}", ha="center", fontsize=8)
ax.set_xticks(range(len(ref)))
ax.set_xticklabels([f"{r.id if r.id.startswith('H') else 'contest'}\n{label.get(r.model, r.model).replace(' (production)', '')}\n"
                    f"{int(r.lines)} l.\n{r.date[5:]}" for r in ref.itertuples()], fontsize=7)
first_contest = next(i for i, r in enumerate(ref.itertuples()) if not r.id.startswith("H"))
ax.axvspan(first_contest - 0.5, len(ref) - 0.5, color="#eef3ee", zorder=0)
ax.text((first_contest + len(ref) - 1) / 2, 30, "contest, 11 Nov", ha="center", fontsize=9, color="#2a8a5a")
ax.set_ylim(0, 32)
ax.set_ylabel("Character error rate (%)")
ax.set_title("All runs on the reference benchmarks, in date order (lower is better)")
ax.text(0, 28.5, "greedy:\nloops", ha="center", fontsize=8, color="#555")

# Right: the contest, accuracy vs speed
ax = axes[1]
label["trocr-base"] = "TrOCR (production)"
offsets = {"trocr-base": (12, -18), "trocr-cursiva": (40, 12), "qwen2-vl-2b-catmus": (16, -4),
           "qwen3-vl-4b-catmus": (-10, 12), "medusa-0.2-4b": (-10, -32)}
for row in contest.itertuples():
    ax.scatter(row.s_per_line, row.CER * 100, s=row.GPU_GB * 60, color=colors.get(row.model, "#888"), alpha=0.8)
    ax.annotate(f"{label.get(row.model, row.model)} {row.CER:.1%}", (row.s_per_line, row.CER * 100),
                textcoords="offset points", xytext=offsets.get(row.model, (10, 5)), fontsize=9,
                ha="right" if offsets.get(row.model, (10, 5))[0] < 0 else ("center" if row.model == "trocr-cursiva" else "left"))
ax.set_xlim(0, 9)
ax.set_ylim(0, 27)
ax.set_xlabel("Seconds per line (Colab T4 GPU)")
ax.set_ylabel("Character error rate (%)")
ax.set_title("Contest: same 100 lines, accuracy vs speed\n(bubble size = GPU memory: 1.4 to 13.7 GB)")

plt.tight_layout()
chart_path = f"{BASE_DIR}/ocr_model_comparison.png"
plt.savefig(chart_path, dpi=150)
plt.show()
print("Chart saved to", chart_path)

## 6. Model registry: which model is in production, and why

Each candidate is registered as a version of the model **`ultron-htr`**. The registry stores a small pointer (the Hugging Face model id and the settings), not the weights.
The alias **`production`** marks the model the app uses; the backend reads it through the `OCR_MODEL_ID` variable.

In [ ]:
import warnings
import mlflow.pyfunc
warnings.filterwarnings("ignore", message="Add type hints")

class HFModelPointer(mlflow.pyfunc.PythonModel):
    """Registry entry for a Hugging Face OCR model: it returns its own settings."""
    def predict(self, context, model_input, params=None):
        return [context.model_config]

REGISTERED = "ultron-htr"
CANDIDATES = [
    dict(model=TROCR_BASE, alias="production", status="production", evidence="H04, H14, H15, C-trocr-base",
         decision="Kept in production: 19.4 % CER in the contest, 1.1 s per line, 1.4 GB GPU. "
                  "The only model that runs a full page quickly on our hardware."),
    dict(model=MEDUSA, alias="candidate", status="candidate: data-leakage check pending",
         evidence="H13, H17, C-medusa-0.2-4b",
         decision="Contest winner: 8.9 % CER vs 19.4 % for TrOCR. Not in production yet: needs a 14 GB GPU and "
                  "6.7 s per line (about 3.5 min for a 30-line page), and it was trained on CATMuS, so we must "
                  "confirm the CATMuS test split was excluded. Next: check the MEDUSA report or test on unseen pages."),
    dict(model=QWEN3_CATMUS, alias=None, status="rejected", evidence="H12, H16, C-qwen3-vl-4b-catmus",
         decision="19.6 % CER, the same as TrOCR, but 7x slower and 10x more GPU memory."),
    dict(model=QWEN2_CATMUS, alias=None, status="rejected", evidence="H06, C-qwen2-vl-2b-catmus",
         decision="22.2 % CER with its training prompt (26.7 % with another prompt), worse than TrOCR."),
    dict(model=TROCR_CURSIVA, alias=None, status="retired", evidence="H07, H08, C-trocr-cursiva",
         decision="22.7 % CER in the contest; replaced by trocr-base in the app."),
]

versions = {}
for mv in client.search_model_versions(f"name='{REGISTERED}'"):
    versions[mv.tags.get("model_id")] = mv.version
for old_alias in ("challenger",):                    # alias used before the contest
    try:
        client.delete_registered_model_alias(REGISTERED, old_alias)
    except Exception:
        pass

for cand in CANDIDATES:
    if cand["model"] in versions:                    # already registered: update its decision
        version = versions[cand["model"]]
        client.update_model_version(REGISTERED, version, description=cand["decision"])
        client.set_model_version_tag(REGISTERED, version, "status", cand["status"])
        client.set_model_version_tag(REGISTERED, version, "evidence_runs", cand["evidence"])
        if cand["alias"]:
            client.set_registered_model_alias(REGISTERED, cand["alias"], version)
        continue
    with mlflow.start_run(run_name=f"registry {short_name(cand['model'])}"):
        mlflow.set_tags({"source": "registry", "model": short_name(cand["model"])})
        config = {"model_id": cand["model"], "status": cand["status"], "evidence_runs": cand["evidence"]}
        try:
            info = mlflow.pyfunc.log_model(name="model", python_model=HFModelPointer(), model_config=config)
        except TypeError:   # MLflow 2.x uses artifact_path instead of name
            info = mlflow.pyfunc.log_model(artifact_path="model", python_model=HFModelPointer(), model_config=config)
    mv = mlflow.register_model(info.model_uri, REGISTERED)
    client.update_model_version(REGISTERED, mv.version, description=cand["decision"])
    client.set_model_version_tag(REGISTERED, mv.version, "model_id", cand["model"])
    client.set_model_version_tag(REGISTERED, mv.version, "status", cand["status"])
    client.set_model_version_tag(REGISTERED, mv.version, "evidence_runs", cand["evidence"])
    if cand["alias"]:
        client.set_registered_model_alias(REGISTERED, cand["alias"], mv.version)

client.update_registered_model(REGISTERED, description=(
    "Line-level handwritten text recognition for medieval manuscripts in Ultron. "
    "Selection rule: switch production only if a model is more than 2 CER points better than the current one "
    "on the same CATMuS benchmark, on test data it has not been trained on, AND runs on our hardware."))

rows = []
for mv in sorted(client.search_model_versions(f"name='{REGISTERED}'"), key=lambda v: int(v.version)):
    aliases = [a for a, v in client.get_registered_model(REGISTERED).aliases.items() if v == mv.version]
    rows.append({"version": mv.version, "model": mv.tags.get("model_id"), "alias": ", ".join(aliases) or "–",
                 "status": mv.tags.get("status"), "evidence": mv.tags.get("evidence_runs"), "decision": mv.description})
display(pd.DataFrame(rows).style.hide(axis="index"))

prod = client.get_model_version_by_alias(REGISTERED, "production")
print(f"\nProduction model: {prod.tags['model_id']}")
print(f"Backend setting:  OCR_MODEL_ID={prod.tags['model_id']}")

## 7. (Optional, GPU) Evaluate a model again and log it as a new run

Needs a GPU runtime. Uses the same 100 lines and the same metric as run H15/H16, and saves the predictions as an artifact of the run.

In [ ]:
RUN_NEW_EVALUATION = False      # set to True, then pick the model below
NEW_MODEL = "trocr"              # "trocr", "qwen3vl_catmus" or "medusa"

if RUN_NEW_EVALUATION:
    %pip install -q -U transformers peft accelerate datasets
    %pip uninstall -q -y torchao
    import gc, itertools, re, time, unicodedata, torch
    from datasets import load_dataset

    ds = load_dataset("CATMuS/medieval", split="test", streaming=True)
    rows300 = list(itertools.islice((r for i, r in enumerate(ds) if i % 50 == 0), 300))
    sample = rows300[::3]
    truths = [r["text"].strip() for r in sample]

    def norm(s): return " ".join(unicodedata.normalize("NFC", str(s)).split())
    def ed(a, b):
        prev = list(range(len(b) + 1))
        for i, x in enumerate(a, 1):
            cur = [i]
            for j, y in enumerate(b, 1):
                cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
            prev = cur
        return prev[-1]
    def rates(ts, ps):
        ce = cn = we = wn = 0
        for t, p in zip(ts, ps):
            t, p = norm(t), norm(p); ce += ed(t, p); cn += len(t); we += ed(t.split(), p.split()); wn += len(t.split())
        return ce / cn, we / wn

    def vlm_reader(model, processor, prompt, extra=None):
        def read(image):
            msgs = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
            inputs = processor.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                                   return_dict=True, return_tensors="pt", **(extra or {})).to(model.device)
            with torch.inference_mode():
                out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
            return " ".join(processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).split())
        return read

    if NEW_MODEL == "trocr":
        from transformers import TrOCRProcessor, VisionEncoderDecoderModel
        proc = TrOCRProcessor.from_pretrained(TROCR_BASE)
        model = VisionEncoderDecoderModel.from_pretrained(TROCR_BASE).to("cuda").eval()
        def read(image):
            pv = proc(images=image.convert("RGB"), return_tensors="pt").pixel_values.to("cuda")
            with torch.inference_mode():
                ids = model.generate(pv, max_new_tokens=128, num_beams=4, no_repeat_ngram_size=4)
            return proc.batch_decode(ids, skip_special_tokens=True)[0].strip()
        model_id, decoding, prompt = TROCR_BASE, BEAMS, "none (TrOCR)"
    elif NEW_MODEL == "qwen3vl_catmus":
        import peft.tuners.lora.torchao as _lt; _lt.is_torchao_available = lambda: False
        from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
        from peft import PeftModel
        processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-4B-Instruct")
        model = Qwen3VLForConditionalGeneration.from_pretrained("Qwen/Qwen3-VL-4B-Instruct", torch_dtype="auto", device_map="auto")
        model = PeftModel.from_pretrained(model, "small-models-for-glam/Qwen3-VL-4B-catmus").eval()
        prompt = "Transcribe the text shown in this image."
        read, model_id, decoding = vlm_reader(model, processor, prompt), QWEN3_CATMUS, GREEDY
    else:
        from transformers import AutoProcessor, AutoModelForImageTextToText
        processor = AutoProcessor.from_pretrained(MEDUSA)
        model = AutoModelForImageTextToText.from_pretrained(MEDUSA, torch_dtype="auto", device_map="auto").eval()
        prompt = "Transcribe the handwritten text in this line image.\nOutput ONLY the transcription."
        read, model_id, decoding = vlm_reader(model, processor, prompt, {"enable_thinking": False}), MEDUSA, GREEDY

    torch.cuda.reset_peak_memory_stats()
    with mlflow.start_run(run_name=f"{short_name(model_id)} catmus-test-every50-third (new)"):
        mlflow.log_params({"model_id": model_id, "model": short_name(model_id), "benchmark": "catmus-test-every50-third",
                           "lines": len(sample), "decoding": decoding, "component": "recognizer",
                           "segmenter": "none (line images)", "prompt": prompt})
        mlflow.set_tags({"source": "new", "date": datetime.now().strftime("%Y-%m-%d")})
        preds, t0 = [], time.time()
        for i, r in enumerate(sample):
            preds.append(read(r["im"].convert("RGB")))
            if (i + 1) % 20 == 0: print(f"{i + 1}/{len(sample)} lines")
        cer, wer = rates(truths, preds)
        mlflow.log_metrics({"cer": cer, "wer": wer, "s_per_line": (time.time() - t0) / len(sample),
                            "gpu_gb": torch.cuda.max_memory_allocated() / 1e9})
        table = pd.DataFrame({"language": [r["language"] for r in sample], "truth": truths, "prediction": preds})
        mlflow.log_table(table, "predictions.json")
        for lang, g in table.groupby("language"):
            mlflow.log_metric(f"cer_{lang.lower()}", rates(g.truth, g.prediction)[0])
        print(f"{short_name(model_id)}: CER {cer:.1%} | WER {wer:.1%}")
    del model; gc.collect(); torch.cuda.empty_cache()
else:
    print("Skipped (set RUN_NEW_EVALUATION = True to evaluate a model).")

## 8. Open the MLflow web interface in Colab

Shows every run, lets you compare runs side by side, and shows the registry with its aliases.

In [ ]:
import subprocess, sys, time
ui = subprocess.Popen([sys.executable, "-m", "mlflow", "ui", "--backend-store-uri", TRACKING_URI, "--port", "5000"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)
try:
    from google.colab import output
    output.serve_kernel_port_as_window(5000, path="/")
except ImportError:
    print("Not in Colab: open http://localhost:5000")

## 9. Export for the report

In [ ]:
runs = load_runs()
runs.to_csv(f"{BASE_DIR}/ocr_experiments.csv", index=False)
print("Saved:", f"{BASE_DIR}/ocr_experiments.csv", "and", chart_path)
try:
    from google.colab import files
    files.download(f"{BASE_DIR}/ocr_experiments.csv")
    files.download(chart_path)
except ImportError:
    pass